# Jubilee digital twin: hands-on

| # | Exercise | Time |
|---|---|---|
| 0 | Setup: a plant on the bed | 10 min |
| 1 | Check the plan before running it | 25 min |
| 2 | Autofocus with the virtual camera | 25 min |
| 3 | Change lighting and background (in Blender) | 20 min |
| 4 | MoGe-2: reconstruct and compare (CloudCompare) | 35 min |

Every step runs **Blender in the background**: the notebook writes a command, Blender opens the scene, does the job, saves files, and quits. Needs Blender on `PATH` and `jubilee-twin setup-scene` run once.

## 0 · Setup

In [ ]:
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

TWIN = Path.cwd().parent                                   # jubilee-blender-twin/
ADDON = TWIN / "blender_addon" / "jubilee_digital_twin"    # the twin's own Blender scripts
PIPE = TWIN / "pipeline_data"                              # where the twin keeps its scenes and CSVs
OUT = Path.cwd() / "out_simple"
OUT.mkdir(exist_ok=True)


def blender(blend, *args):
    # Run Blender without a window on `blend`, return what it printed.
    cmd = ["blender", str(blend), "--background", *map(str, args)]
    run = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if run.returncode != 0:
        print(run.stdout[-3000:], run.stderr[-3000:])
        raise RuntimeError("Blender failed")
    return run.stdout


def show(*paths, titles=None, width=5):
    fig, axes = plt.subplots(1, len(paths), figsize=(width * len(paths), width * 0.6), squeeze=False)
    for i, (ax, p) in enumerate(zip(axes[0], paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles: ax.set_title(titles[i])
    plt.tight_layout()

The working scene has the machine and the calibrated `Toolhead_Cam`, but nothing on the bed. [`scripts/build_scene.py`](scripts/build_scene.py) adds the basil from `basil_3D.blend` onto the bed plate (so it moves with Z) and gives the camera a real lens with a fixed focus distance.

In [ ]:
SCENE = PIPE / "workshop_simple.blend"
log = blender(PIPE / "jubilee_working.blend", "--python", "scripts/build_scene.py", "--",
              TWIN / "blender_models" / "basil_3D.blend", SCENE)
print("Scene saved:", SCENE)

First picture, with the twin's own `snapshot.py`. **X/Y** move the camera, **Z** moves the bed: bigger Z means the bed is lower.

In [ ]:
blender(SCENE, "--python", ADDON / "snapshot.py", "--", "--x", 147, "--y", 145, "--z", 310,
        "--output", OUT / "first.jpg")
show(OUT / "first.jpg", width=8)

---
## 1 · Check the plan before running it

This raster was written for a flat Petri dish, with the bed raised to Z = 150. Today the basil is on the bed. **What will happen?**

In [ ]:
PLAN = '''G90
G0 X100 Y100 Z150
G1 X200 Y100
G1 X200 Y145
G1 X100 Y145
G1 X100 Y190
G1 X200 Y190'''

**Step 1: G-code → path.** The twin's parser turns the plan into one machine position every 10 mm.
It always starts at home (0, 0, 0), which with this plant is *inside* the leaves. On the real machine you load the plant with the bed already lowered, so we start at the first commanded position.

In [ ]:
import contextlib, io
from jubilee_twin.pipeline.path_follower import build_path

def plan_to_path(plan, step_mm=10):
    lines = plan.splitlines()
    with contextlib.redirect_stdout(io.StringIO()):          # the parser prints debug lines
        path = np.array(build_path(lines, step_mm))
        to_start = build_path(lines[:2], step_mm)            # home -> first position
    path = path[len(to_start) - 1:]
    np.savetxt(PIPE / "pathout.csv", path, delimiter=",", fmt="%.3f")   # the file the twin animates
    return path

path = plan_to_path(PLAN)
print(len(path), "frames, first ones:\n", path[:4, :3])

**Step 2: animate + ray-trace**, with the twin's `animate_path.py` and `ray_tracing.py`, on a copy of the scene. Blender saves the animated copy, so you can open it and watch.

In [ ]:
def check(plan, name):
    path = plan_to_path(plan)
    saved = PIPE / f"check_{name}.blend"
    shutil.copy(SCENE, saved)
    log = blender(saved, "--python", ADDON / "animate_path.py", "--python", ADDON / "ray_tracing.py")
    hits = [(int(frame), obj) for obj, frame in re.findall(r"Collision between gantry and (.+) during frame (\d+)", log)]
    print(f"{len(hits)} collision frame(s)")
    for frame, obj in hits[:5]:
        x, y, z = path[frame - 1, :3]
        print(f"  frame {frame:3d}: gantry hits {obj!r} at X={x:.0f} Y={y:.0f} Z={z:.0f}")
    print(f"\nAnimated scene saved: {saved}")
    print("Open it in Blender, press Space to play, or jump to the frames listed above.")
    return hits, saved

hits, saved = check(PLAN, "petri_plan")

In [ ]:
# subprocess.Popen(["blender", str(saved)])   # uncomment: opens the animated check in the Blender GUI

### ✏️ Your turn
Fix the plan so it is safe and re-check it. Keep the camera as close as you safely can. Bonus: what is the lowest safe Z?

In [ ]:
FIXED = PLAN.replace("Z150", "Z150")    # ✏️
hits, saved = check(FIXED, "fixed_plan")

---
## 2 · Autofocus with the virtual camera

The camera has a fixed focus distance, so moving the bed in Z moves the plant in and out of focus. We take a **Z-stack** with the twin's virtual scanner (`scan.py`, the same as `jubilee-twin scan`), then let a sharpness score pick the best Z.

In [ ]:
X, Y = 147, 145
blender(SCENE, "--python", ADDON / "scan.py", "--",
        "--x-min", X, "--x-max", X, "--x-steps", 1,
        "--y-min", Y, "--y-max", Y, "--y-steps", 1,
        "--z-min", 220, "--z-max", 340, "--z-steps", 9,
        "--width", 960, "--height", 528, "--output-root", OUT / "zstack")

scan_dir = sorted((OUT / "zstack").iterdir())[-1]           # newest scan folder
images = sorted(scan_dir.glob("*.jpg"), key=lambda p: int(p.stem.split("_z")[1]))
zs = [int(p.stem.split("_z")[1]) for p in images]
print("Saved in", scan_dir)

**Sharpness = variance of the Laplacian**: a sharp image has strong edges, so its second derivative varies a lot.

In [ ]:
from scipy import ndimage

def sharpness(path):
    gray = np.asarray(Image.open(path).convert("L"), float)
    return ndimage.laplace(gray).var()

scores = [sharpness(p) for p in images]
best = int(np.argmax(scores))
print("Best focus: Z =", zs[best])

plt.figure(figsize=(5, 3)); plt.plot(zs, scores, "-o"); plt.axvline(zs[best], color="crimson", ls="--")
plt.xlabel("bed Z (mm)"); plt.ylabel("sharpness")
show(images[0], images[best], titles=[f"Z = {zs[0]}", f"Z = {zs[best]} (chosen)"])

### ✏️ Your turn
- A Z-stack costs time and light. Find the same best Z with **fewer images** (coarse then fine).
- Same scanner, other use: a **raster** over the plant (`--x-steps 3 --y-steps 3`, one Z).

In [ ]:
# ✏️

---
## 3 · Change lighting and background, in Blender

The scene is lit by an **HDRI**, a 360° photo that is *both* the light and the background: the wooden floor in the renders is the kitchen HDRI. Two HDRIs are in `workshop/hdri/`; add your own from [polyhaven.com/hdris](https://polyhaven.com/hdris) (1k `.hdr` is enough).

Open the scene:

In [ ]:
subprocess.Popen(["blender", str(SCENE)])
print("Best focus from Exercise 2: Z =", zs[best])

**1. Put the camera above the plant.** Press **N** in the 3D view → **Twin** tab → *Move to Position*: X 147, Y 145, Z = the best focus from Exercise 2.

**2. Look through the toolhead camera.** In the Outliner, click `Toolhead_Cam`, then **View → Cameras → Set Active Object as Camera** (Ctrl+Numpad 0). Switch the viewport to **Rendered** shading (top-right sphere icons, or **Z** → Rendered): lighting now updates live.

**3. Change the HDRI.** Properties panel → **World** tab (red globe). Under *Surface*, the *Color* is an **Environment Texture** showing the kitchen file: click the folder icon next to it and open an `.hdr` from `workshop/hdri/`. The floor and the light change together.

**4. Change the intensity.** Same panel, *Strength*: 3 is the twin's default, try 0.5 and 8.

**5. Rotate the light.** Top tabs → **Shading** workspace. In the node editor, switch the header from *Object* to **World**. Enable *Node Wrangler* once (Edit → Preferences → Add-ons → search "Node Wrangler"), click the Environment Texture node and press **Ctrl+T**: a *Mapping* node appears. Change *Rotation Z* and watch the shadows move around the leaves.

**6. Render.** **F12** renders through the toolhead camera; *Image → Save As* to keep it. (Or *Take Snapshot* in the Twin panel, saved to `Scans/snapshots/`.)

Repeat with the other HDRI and compare. The plant, the camera, the geometry did not move: only the appearance did. So every look shares **the same depth, the same masks, the same camera pose**: labels come for free.

> **Ctrl+S** saves your lighting into the scene. Exercise 4 will then render with your look.

---
## 4 · MoGe-2: reconstruct from one image, compare in CloudCompare

[MoGe-2](https://huggingface.co/spaces/Ruicheng/MoGe-2) predicts a 3D point for every pixel of a single image. On a real plant nobody knows the true shape. In the twin we do: [`scripts/ground_truth.py`](scripts/ground_truth.py) renders the image and shoots one ray per pixel to record the true plant surface the camera sees.

In [ ]:
EX4 = OUT / "moge"
log = blender(SCENE, "--python", "scripts/ground_truth.py", "--", X, Y, zs[best], EX4)   # ~30 s
print([line for line in log.splitlines() if "twin point cloud" in line][0])
show(EX4 / "moge_input.png", width=8)
print("Upload this image:", EX4 / "moge_input.png")

### Run MoGe-2 online
1. Open **https://huggingface.co/spaces/Ruicheng/MoGe-2**, upload `out_simple/moge/moge_input.png`, run.
2. In the *Download* tab, save **`pointcloud.ply`** right away (the demo deletes its files after a few minutes) into `out_simple/moge/`.

### Compare in CloudCompare
Both files are in metres, with the camera at the origin and the same axes, so they open already aligned.

1. **File → Open** `pointcloud.ply` (MoGe) and `twin_plant.ply` (the truth). Tick/untick them in the DB tree to compare.
2. **Keep only the plant in MoGe's cloud.** Select `pointcloud`, rotate to a side view (the bed is a flat sheet behind the plant), **Edit → Segment** (scissors), draw a polygon around the plant, right-click to close, *Segment In*, then the green tick.
3. **Is MoGe metric?** Select both clouds → **Tools → Registration → Fine Registration (ICP)**. Reference = `twin_plant`, aligned = MoGe's plant, tick **Adjust scale**. The report gives the scale factor: 1.00 would be perfect metric scale.
4. **How good is the shape?** **Tools → Distances → Cloud/Cloud Dist.**, compared = MoGe's plant, reference = `twin_plant`, *Compute*. MoGe's cloud is now coloured by its distance to the true surface (in metres: 0.005 = 5 mm). **Edit → Scalar fields → Show histogram** for the distribution.

### ✏️ Discussion
- Where is MoGe wrong: leaf edges, overlapping leaves, soil? Why?
- Which matters more for collision checking: the scale error or the shape error?
- Render again after changing the HDRI (Exercise 3, Ctrl+S): the truth is the same, does MoGe's answer change?